# Model Evaluation Notebook

This notebook evaluates the sliding-window, YOLO, and RF-DETR models on the Gunmen dataset using:

1. mAP@[0.5:0.95]
2. Precision and recall per class
3. Confusion matrix
4. Precision-recall curves
5. Inference speed

The notebook uses the repository's existing model-loading and preprocessing code paths so the metrics line up with the project's training and inference setup.

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import Any

import fiddle as fdl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.ops import nms
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

from src.config.constants import Constants
from src.config.sliding_window import build_config as build_sliding_window_config
from src.config.yolo_detection import build_config as build_yolo_config
from src.config.rfdetr_detection import build_config as build_rfdetr_config
from src.datasets.gunmen_dataset import GunmenYoloDataset
from src.utils.config import parse_fiddle_config

plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "sources" / "Gunmen Dataset" / "All"
DEFAULT_DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

SLIDING_WINDOW_CONFIG = PROJECT_ROOT / "src" / "config" / "sliding_window.py"
Yolo_CONFIG = PROJECT_ROOT / "src" / "config" / "yolo_detection.py"
RFDETR_CONFIG = PROJECT_ROOT / "src" / "config" / "rfdetr_detection.py"

SLIDING_WINDOW_CKPT: Path | None = None
YOLO_CKPT = (
    PROJECT_ROOT
    / "logs"
    / "gunmen_yolo_finetune_20260521_162611"
    / "epoch=17-step=2358.ckpt"
)
RFDETR_CKPT = (
    PROJECT_ROOT
    / "logs"
    / "gunmen_rfdetr_finetune_20260601_212051"
    / "epoch=0-step=262.ckpt"
)

EVAL_MAX_IMAGES: int | None = None
SPEED_SAMPLE_COUNT = 16
IOU_THRESHOLDS = [round(value, 2) for value in np.arange(0.5, 1.0, 0.05)]
MATCH_IOU_THRESHOLD = 0.5
CLASS_NAMES = ["human", "gun"]
BACKGROUND_INDEX = len(CLASS_NAMES)


## Utils

In [ ]:
@dataclass(frozen=True)
class EvaluationSample:
    image_path: Path
    image_size: tuple[int, int]
    gt_boxes: torch.Tensor
    gt_labels: torch.Tensor


def sync_device(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def xywh_to_xyxy(boxes: torch.Tensor, image_size: tuple[int, int]) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)

    width, height = image_size
    x_center = boxes[:, 0] * width
    y_center = boxes[:, 1] * height
    box_width = boxes[:, 2] * width
    box_height = boxes[:, 3] * height

    x1 = x_center - box_width / 2
    y1 = y_center - box_height / 2
    x2 = x_center + box_width / 2
    y2 = y_center + box_height / 2
    return torch.stack((x1, y1, x2, y2), dim=1)


def clip_boxes(boxes: torch.Tensor, image_size: tuple[int, int]) -> torch.Tensor:
    if boxes.numel() == 0:
        return boxes.reshape(0, 4)

    width, height = image_size
    clipped = boxes.clone()
    clipped[:, 0::2] = clipped[:, 0::2].clamp(0, width)
    clipped[:, 1::2] = clipped[:, 1::2].clamp(0, height)
    return clipped


def box_iou(box: torch.Tensor, boxes: torch.Tensor) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.zeros((0,), dtype=torch.float32)

    inter_x1 = torch.maximum(box[0], boxes[:, 0])
    inter_y1 = torch.maximum(box[1], boxes[:, 1])
    inter_x2 = torch.minimum(box[2], boxes[:, 2])
    inter_y2 = torch.minimum(box[3], boxes[:, 3])

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter_area = inter_w * inter_h

    box_area = (box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0)
    boxes_area = (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (
        boxes[:, 3] - boxes[:, 1]
    ).clamp(min=0)

    union = box_area + boxes_area - inter_area
    return torch.where(union > 0, inter_area / union, torch.zeros_like(inter_area))


def load_samples(
    dataset_root: Path, max_images: int | None = None
) -> list[EvaluationSample]:
    dataset = GunmenYoloDataset(dataset_root=dataset_root, strict=False)
    sample_indices = list(range(len(dataset)))
    if max_images is not None:
        sample_indices = sample_indices[:max_images]

    samples: list[EvaluationSample] = []
    for index in sample_indices:
        image_path = dataset.get_sample_path(index)
        label_path = dataset.get_raw_label_path(index)
        image = Image.open(image_path).convert("RGB")
        image_size = image.size
        del image

        targets = dataset._parse_yolo_label_file(label_path)
        if targets.numel() == 0:
            gt_boxes = torch.zeros((0, 4), dtype=torch.float32)
            gt_labels = torch.zeros((0,), dtype=torch.long)
        else:
            gt_labels = targets[:, 0].to(dtype=torch.long)
            gt_boxes = xywh_to_xyxy(targets[:, 1:5], image_size)

        samples.append(
            EvaluationSample(
                image_path=image_path,
                image_size=image_size,
                gt_boxes=gt_boxes,
                gt_labels=gt_labels,
            )
        )

    return samples


def classwise_nms(
    boxes: torch.Tensor,
    scores: torch.Tensor,
    labels: torch.Tensor,
    iou_threshold: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if boxes.numel() == 0:
        return boxes.reshape(0, 4), scores.reshape(0), labels.reshape(0)

    keep_indices: list[torch.Tensor] = []
    for class_id in labels.unique(sorted=True):
        class_mask = labels == class_id
        class_keep = nms(boxes[class_mask], scores[class_mask], iou_threshold)
        class_indices = torch.nonzero(class_mask, as_tuple=False).flatten()[class_keep]
        keep_indices.append(class_indices)

    if not keep_indices:
        return boxes.reshape(0, 4), scores.reshape(0), labels.reshape(0)

    keep = torch.cat(keep_indices)
    keep = keep[scores[keep].argsort(descending=True)]
    return boxes[keep], scores[keep], labels[keep]


def load_lightning_model(
    config_path: Path, ckpt_path: Path, device: torch.device
) -> tuple[Any, torch.nn.Module]:
    cfg = parse_fiddle_config(config_path)
    built_cfg = fdl.build(cfg)
    model = built_cfg.model
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    model = model.to(device)
    model.eval()
    return built_cfg, model


def predict_sliding_window_image(
    model: torch.nn.Module,
    image: Image.Image,
    crop_size: int,
    device: torch.device,
    window_size: int = 128,
    stride: int = 32,
    threshold: float = 0.8,
    iou_threshold: float = 0.1,
) -> dict[str, torch.Tensor]:
    transform = T.Compose([T.Resize((crop_size, crop_size)), T.ToTensor()])
    image_width, image_height = image.size

    boxes: list[tuple[float, float, float, float]] = []
    scores: list[float] = []
    labels: list[int] = []

    with torch.inference_mode():
        for top in range(0, max(1, image_height - window_size + 1), stride):
            for left in range(0, max(1, image_width - window_size + 1), stride):
                box = (
                    float(left),
                    float(top),
                    float(left + window_size),
                    float(top + window_size),
                )
                crop = image.crop(tuple(int(value) for value in box))
                crop_tensor = transform(crop).unsqueeze(0).to(device)
                logits = model(crop_tensor)
                probabilities = torch.softmax(logits, dim=1)[0]
                class_id = int(torch.argmax(probabilities).item())
                score = float(probabilities[class_id].item())

                if class_id in (1, 2) and score >= threshold:
                    boxes.append(box)
                    scores.append(score)
                    labels.append(class_id - 1)

    if not boxes:
        return {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros((0,), dtype=torch.float32),
            "labels": torch.zeros((0,), dtype=torch.long),
        }

    box_tensor = torch.tensor(boxes, dtype=torch.float32)
    score_tensor = torch.tensor(scores, dtype=torch.float32)
    label_tensor = torch.tensor(labels, dtype=torch.long)
    box_tensor, score_tensor, label_tensor = classwise_nms(
        box_tensor, score_tensor, label_tensor, iou_threshold
    )
    return {
        "boxes": box_tensor.cpu(),
        "scores": score_tensor.cpu(),
        "labels": label_tensor.cpu(),
    }


def predict_yolo_image(
    model: torch.nn.Module,
    image: Image.Image,
    device: torch.device,
    image_size: int = 640,
    confidence_threshold: float = 0.001,
    iou_threshold: float = 0.45,
) -> dict[str, torch.Tensor]:
    image_array = np.asarray(image)
    resized_array = LetterBox(new_shape=(image_size, image_size), auto=False)(
        image=image_array
    )
    tensor = (
        torch.from_numpy(np.ascontiguousarray(resized_array))
        .permute(2, 0, 1)
        .float()
        .div(255.0)
        .unsqueeze(0)
    )
    tensor = tensor.to(device)

    with torch.inference_mode():
        predictions = model(tensor)[0]
        detections = non_max_suppression(
            predictions,
            conf_thres=confidence_threshold,
            iou_thres=iou_threshold,
            nc=getattr(model, "nc", 2),
        )[0]

    if detections is None or detections.numel() == 0:
        return {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros((0,), dtype=torch.float32),
            "labels": torch.zeros((0,), dtype=torch.long),
        }

    boxes = detections[:, :4].detach().cpu()
    boxes = scale_boxes(resized_array.shape[:2], boxes, image.size[::-1]).round()
    scores = detections[:, 4].detach().cpu()
    labels = detections[:, 5].detach().cpu().to(torch.long)
    return {"boxes": boxes, "scores": scores, "labels": labels}


def predict_rfdetr_image(
    model: torch.nn.Module,
    image: Image.Image,
    device: torch.device,
    confidence_threshold: float = 0.001,
    iou_threshold: float = 0.45,
) -> dict[str, torch.Tensor]:
    inputs = model.image_processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.detector(**inputs)

    results = model.image_processor.post_process_object_detection(
        outputs,
        target_sizes=torch.tensor([image.size[::-1]], device=device),
        threshold=confidence_threshold,
    )[0]

    boxes = results["boxes"].detach().cpu()
    scores = results["scores"].detach().cpu()
    labels = results["labels"].detach().cpu().to(torch.long)
    boxes, scores, labels = classwise_nms(boxes, scores, labels, iou_threshold)
    return {"boxes": boxes, "scores": scores, "labels": labels}


### plot utils

In [ ]:
def evaluate_confusion_matrix(
    samples: list[EvaluationSample],
    predictions: list[dict[str, torch.Tensor]],
    iou_threshold: float = MATCH_IOU_THRESHOLD,
) -> np.ndarray:
    matrix = np.zeros((len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1), dtype=np.int64)

    for sample, prediction in zip(samples, predictions):
        gt_boxes = sample.gt_boxes
        gt_labels = sample.gt_labels
        pred_boxes = prediction["boxes"]
        pred_scores = prediction["scores"]
        pred_labels = prediction["labels"]

        matched_gt: set[int] = set()
        order = pred_scores.argsort(descending=True)
        for pred_index in order.tolist():
            pred_box = pred_boxes[pred_index]
            pred_label = int(pred_labels[pred_index].item())
            if gt_boxes.numel() == 0:
                matrix[BACKGROUND_INDEX, pred_label] += 1
                continue

            ious = box_iou(pred_box, gt_boxes)
            if ious.numel() == 0:
                matrix[BACKGROUND_INDEX, pred_label] += 1
                continue

            best_iou, best_index = torch.max(ious, dim=0)
            best_gt_index = int(best_index.item())
            if (
                float(best_iou.item()) >= iou_threshold
                and best_gt_index not in matched_gt
            ):
                gt_label = int(gt_labels[best_gt_index].item())
                matrix[gt_label, pred_label] += 1
                matched_gt.add(best_gt_index)
            else:
                matrix[BACKGROUND_INDEX, pred_label] += 1

        for gt_index, gt_label in enumerate(gt_labels.tolist()):
            if gt_index not in matched_gt:
                matrix[gt_label, BACKGROUND_INDEX] += 1

    return matrix


def precision_recall_from_confusion_matrix(matrix: np.ndarray) -> pd.DataFrame:
    rows = []
    for class_index, class_name in enumerate(CLASS_NAMES):
        true_positive = matrix[class_index, class_index]
        false_positive = matrix[:, class_index].sum() - true_positive
        false_negative = matrix[class_index, :].sum() - true_positive
        precision = (
            true_positive / (true_positive + false_positive)
            if (true_positive + false_positive)
            else 0.0
        )
        recall = (
            true_positive / (true_positive + false_negative)
            if (true_positive + false_negative)
            else 0.0
        )
        rows.append(
            {
                "class": class_name,
                "precision": precision,
                "recall": recall,
                "tp": int(true_positive),
                "fp": int(false_positive),
                "fn": int(false_negative),
            }
        )
    return pd.DataFrame(rows)


def precision_envelope(precisions: np.ndarray) -> np.ndarray:
    return np.maximum.accumulate(precisions[::-1])[::-1]


def compute_pr_curve_for_class(
    samples: list[EvaluationSample],
    predictions: list[dict[str, torch.Tensor]],
    class_id: int,
    iou_threshold: float,
) -> tuple[np.ndarray, np.ndarray, float]:
    detections: list[tuple[float, int, torch.Tensor]] = []
    gt_by_image: list[torch.Tensor] = []

    for sample_index, (sample, prediction) in enumerate(zip(samples, predictions)):
        gt_mask = sample.gt_labels == class_id
        gt_by_image.append(sample.gt_boxes[gt_mask])
        pred_mask = prediction["labels"] == class_id
        for box, score in zip(
            prediction["boxes"][pred_mask], prediction["scores"][pred_mask]
        ):
            detections.append((float(score.item()), sample_index, box))

    total_gt = sum(boxes.shape[0] for boxes in gt_by_image)
    if total_gt == 0:
        recall = np.array([0.0])
        precision = np.array([0.0])
        return recall, precision, 0.0

    if not detections:
        recall = np.array([0.0])
        precision = np.array([0.0])
        return recall, precision, 0.0

    detections.sort(key=lambda item: item[0], reverse=True)
    matched: list[set[int]] = [set() for _ in samples]
    true_positives: list[int] = []
    false_positives: list[int] = []

    for score, sample_index, predicted_box in detections:
        gt_boxes = gt_by_image[sample_index]
        if gt_boxes.numel() == 0:
            true_positives.append(0)
            false_positives.append(1)
            continue

        ious = box_iou(predicted_box, gt_boxes)
        if ious.numel() == 0:
            true_positives.append(0)
            false_positives.append(1)
            continue

        best_iou, best_index = torch.max(ious, dim=0)
        best_gt_index = int(best_index.item())
        if (
            float(best_iou.item()) >= iou_threshold
            and best_gt_index not in matched[sample_index]
        ):
            matched[sample_index].add(best_gt_index)
            true_positives.append(1)
            false_positives.append(0)
        else:
            true_positives.append(0)
            false_positives.append(1)

    tp_cumulative = np.cumsum(true_positives)
    fp_cumulative = np.cumsum(false_positives)
    recall = tp_cumulative / total_gt
    precision = tp_cumulative / np.maximum(tp_cumulative + fp_cumulative, 1)
    precision = precision_envelope(precision)

    recall_points = np.linspace(0.0, 1.0, 101)
    interpolated_precision = np.interp(
        recall_points, recall, precision, left=0.0, right=0.0
    )
    ap = float(interpolated_precision.mean())
    return recall, precision, ap


def compute_map_5095(
    samples: list[EvaluationSample],
    predictions: list[dict[str, torch.Tensor]],
) -> tuple[
    float, pd.DataFrame, dict[str, dict[float, tuple[np.ndarray, np.ndarray, float]]]
]:
    curve_store: dict[str, dict[float, tuple[np.ndarray, np.ndarray, float]]] = {
        name: {} for name in CLASS_NAMES
    }
    ap_rows: list[dict[str, float]] = []

    for class_id, class_name in enumerate(CLASS_NAMES):
        ap_values: list[float] = []
        for iou_threshold in IOU_THRESHOLDS:
            recall, precision, ap = compute_pr_curve_for_class(
                samples, predictions, class_id, iou_threshold
            )
            curve_store[class_name][iou_threshold] = (recall, precision, ap)
            ap_values.append(ap)

        ap_rows.append(
            {
                "class": class_name,
                **{
                    f"ap@{iou_threshold:.2f}": curve_store[class_name][iou_threshold][2]
                    for iou_threshold in IOU_THRESHOLDS
                },
                "ap@0.5:0.95": float(np.mean(ap_values)),
            }
        )

    ap_frame = pd.DataFrame(ap_rows)
    map_5095 = float(ap_frame["ap@0.5:0.95"].mean())
    return map_5095, ap_frame, curve_store


def plot_confusion_matrix(matrix: np.ndarray, title: str) -> None:
    labels = [*CLASS_NAMES, "background"]
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels=labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Ground truth")
    ax.set_title(title)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            ax.text(
                column_index,
                row_index,
                int(matrix[row_index, column_index]),
                ha="center",
                va="center",
                color="black",
            )

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


def plot_pr_curves(
    curves: dict[str, dict[float, tuple[np.ndarray, np.ndarray, float]]], title: str
) -> None:
    fig, axes = plt.subplots(
        1, len(CLASS_NAMES), figsize=(14, 5), sharex=True, sharey=True
    )
    if len(CLASS_NAMES) == 1:
        axes = [axes]

    for axis, class_name in zip(axes, CLASS_NAMES):
        recall, precision, ap = curves[class_name][0.5]
        axis.plot(recall, precision, linewidth=2, label=f"AP@0.5={ap:.3f}")
        axis.set_title(class_name)
        axis.set_xlabel("Recall")
        axis.set_ylabel("Precision")
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1.05)
        axis.legend(loc="lower left")
        axis.grid(True, alpha=0.3)

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def measure_speed(
    predictor: Any,
    samples: list[EvaluationSample],
    device: torch.device,
) -> dict[str, float]:
    timings: list[float] = []
    images = [
        Image.open(sample.image_path).convert("RGB")
        for sample in samples[:SPEED_SAMPLE_COUNT]
    ]

    for image in images:
        sync_device(device)
        start = perf_counter()
        predictor(image)
        sync_device(device)
        timings.append(perf_counter() - start)

    if not timings:
        return {"mean_ms": 0.0, "fps": 0.0}

    mean_seconds = float(np.mean(timings))
    return {
        "mean_ms": mean_seconds * 1000.0,
        "fps": 1.0 / mean_seconds if mean_seconds else 0.0,
    }


## Plots

In [6]:
samples = load_samples(DATASET_ROOT, max_images=EVAL_MAX_IMAGES)
yolo_cfg, yolo_model = load_lightning_model(Yolo_CONFIG, YOLO_CKPT, DEFAULT_DEVICE)
rfdetr_cfg, rfdetr_model = load_lightning_model(
    RFDETR_CONFIG, RFDETR_CKPT, DEFAULT_DEVICE
)
# sliding_window_predictor = lambda image: predict_sliding_window_image(
#     model=yolo_model,
#     image=image,
#     crop_size=yolo_cfg.data.image_size,
#     device=DEFAULT_DEVICE,
# )
yolo_predictor = lambda image: predict_yolo_image(
    model=yolo_model,
    image=image,
    device=DEFAULT_DEVICE,
)
rfdetr_predictor = lambda image: predict_rfdetr_image(
    model=rfdetr_model,
    image=image,
    device=DEFAULT_DEVICE,
)
# sliding_window_predictions = [
#     sliding_window_predictor(sample.image_path) for sample in samples
# ]
yolo_predictions = [yolo_predictor(sample.image_path) for sample in samples]
rfdetr_predictions = [rfdetr_predictor(sample.image_path) for sample in samples]
# sliding_window_confusion = evaluate_confusion_matrix(
#     samples, sliding_window_predictions
# )
yolo_confusion = evaluate_confusion_matrix(samples, yolo_predictions)
rfdetr_confusion = evaluate_confusion_matrix(samples, rfdetr_predictions)
# sliding_window_map, sliding_window_ap_frame, sliding_window_curves = compute_map_5095(
#     samples, sliding_window_predictions
# )
yolo_map, yolo_ap_frame, yolo_curves = compute_map_5095(samples, yolo_predictions)
rfdetr_map, rfdetr_ap_frame, rfdetr_curves = compute_map_5095(
    samples, rfdetr_predictions
)
# print("Sliding Window AP Frame:")
# print(sliding_window_ap_frame)
print("\nYOLO AP Frame:")
print(yolo_ap_frame)
print("\nRFDETR AP Frame:")
print(rfdetr_ap_frame)
# plot_confusion_matrix(sliding_window_confusion, "Sliding Window Confusion Matrix")
plot_confusion_matrix(yolo_confusion, "YOLO Confusion Matrix")
plot_confusion_matrix(rfdetr_confusion, "RFDETR Confusion Matrix")
# plot_pr_curves(sliding_window_curves, "Sliding Window PR Curves")
plot_pr_curves(yolo_curves, "YOLO PR Curves")
plot_pr_curves(rfdetr_curves, "RFDETR PR Curves")


Overriding model.yaml nc=80 with nc=2
Transferred 319/355 items from pretrained weights


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `91`.


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

[transformers] RfDetrForObjectDetection LOAD REPORT from: Roboflow/rf-detr-medium
Key                                       | Status   |                                                                                        
------------------------------------------+----------+----------------------------------------------------------------------------------------
model.enc_out_class_embed.{0...12}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91, 256]) vs model:torch.Size([2, 256])
class_embed.weight                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91, 256]) vs model:torch.Size([2, 256])
class_embed.bias                          | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91]) vs model:torch.Size([2])          
model.enc_out_class_embed.{0...12}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([91]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did n

IndexError: tuple index out of range

### Speed

In [ ]:
sliding_window_speed = measure_speed(sliding_window_predictor, samples, DEFAULT_DEVICE)
yolo_speed = measure_speed(yolo_predictor, samples, DEFAULT_DEVICE)
rfdetr_speed = measure_speed(rfdetr_predictor, samples, DEFAULT_DEVICE)
print(f"Sliding Window Speed: {sliding_window_speed}")
print(f"YOLO Speed: {yolo_speed}")
print(f"RFDETR Speed: {rfdetr_speed}")
